# ShallowLandslider: Quick-start Notebook

This notebook runs the `ShallowLandslider` modelling component on a Landlab grid using bundled example data by default. It demonstrates DEM setup, flow routing, soil-depth assignment, earthquake PGA forcing, landslide selection, optional Newmark displacement, optional runout, and publication-ready summary plots.

To create a compatible environment, and install all deps:
```bash
conda env create -f environment.yml
conda activate shallow_landslider
```


## Prerequisites

- Python 3.9+ recommended
- Packages: `numpy`$\geq$ 2.0, `matplotlib`, `pandas`, `landlab`, `scipy`, `scikit-image`, `richdem`, `seaborn`, 
- Local modules available on your path: `components.shallow_landslider`, and `utils` providing:
  - `get_topo`, `apply_soil_depth`, `calculate_terrain_attribute`, `generate_acceleration_grid`,
  - `pickle_or_not_to_pickle` (if using measured data)

> If running in a clean environment, install deps, e.g.:
```bash
pip install landlab numpy pandas matplotlib scipy scikit-image seaborn richdem
```

> Ensure the working directory contains your component and helper modules, or update `sys.path` accordingly.
> The component has been tested on RasterModelGrid **only**; it might not work with other types of grids

## Imports & Settings

In [ ]:
# Standard imports
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Landlab
from landlab.components import PriorityFloodFlowRouter
from landlab import imshowhs_grid

# Project modules
from components.shallow_landslider import ShallowLandslider

# Utilities to help quick-start
from utils import (
    get_topo,
    apply_soil_depth,
    calculate_terrain_attribute,
    generate_acceleration_grid,
    pickle_or_not_to_pickle,
    plot_comparison_panels_with_ecdf,
    save_model_run,
    load_all_runs,
    setup_logger,
)

# Plotting aesthetics
plt.style.use('seaborn-v0_8')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.grid': False,
})


## Configuration 1 (edit these for your study area)

In [ ]:
# DEM & flow settings
config = {
    'dem_info': {
        'region': 'nepal',  # 'nepal', 'png', 'nz', 'japan'
        'dem_type': 'SRTMGL1',
        'buffer': 0.01,
        'smooth_num': 4,
        'plot_dem': True,
        # Use a bundled DEM by default so the public notebook runs without network access.
        'use_local_dem': True,
        'local_dem_path': './input_data/dem/SRTMGL1_28.169999999999998_85.03_28.3_85.21000000000001.asc',
        # Optional: set OPENTOPOGRAPHY_API_KEY in your environment to download a fresh DEM.
        'api_key': os.environ.get('OPENTOPOGRAPHY_API_KEY'),
        # Bounding box used only when use_local_dem=False.
        'north': 28.29,
        'east': 85.20,
        'south': 28.18,
        'west': 85.04,
    },
    'flow_params': {
        'flow_metric': 'D8',
        'separate_hill_flow': True,  # Required for optional runout; creates hill-flow receiver fields
        'depression_handling': 'fill',
        'update_hill_depressions': True,
        'accumulate_flow': True,
    },
    'soil_params': {  # Edit these soil properties
        'angle_int_frict': 30,      # degrees
        'cohesion_eff': 15e3,       # Pa
        'submerged_soil_proportion': 0.5,
        'max_soil_depth': 1.5,      # m
        'plot_soil': True,
        'distribution': 'curvature', # 'uniform'|'elevation'|'drainage_area'|'curvature'|'mean_elev_curv'
        'relationship': 'linear_std_local',
        'decay_rate': 1.0,
        'exponent': 2.0,
        'drainage_transform': 'threshold',
        'drainage_threshold': 1e6,
        'drainage_power': 0.3,
        'P0': 0.05,
        'h_star': 1.0,
        'D': 0.01,
        'h_min': 0.1,
        'h_no_ss': 0.0,
    },
    'pga': {
        'horizontal_max': 0.6,
        'vertical_max': 0.2,
        'distribution': 'uniform', # 'uniform', 'circular', 'square', 'diamond', or 'exponential'
        'plot_grids': False,
    },
    'simulation': {
        'time_shaking': 10, # seconds; used for Newmark displacement if enabled
        'displacement_threshold': 0.0,
        'aspect_interval': 20,
        'random_seed': 5000,
        'handle_small_regions': 'merge',
        'selection_method': 'probabilistic', # or 'pga_weighted'
        'proportion_method': 'conservative',
        'custom_proportion': 0.25,
        'compute_displacement': True, # Required for Newmark displacement and optional runout
        'enable_runout': False,       # Set True with update_soil=True to route failed material downslope
        'update_soil': False,         # Required with enable_runout=True; runout modifies soil__depth in place
    },
    'output': {
        'verbose': True,
        'save_plots': False,
        'save_output_pickle': False, # Keep False for a clean public quickstart; set True to compare runs later.
    }
}


## Load measured landslide data for KDE‑guided splitting
- pickle_or_not_to_pickle bundles all of the input measured data for quick access, and allows the model to pick up the required KDE data for splitting later.
    - Once a pickle has been made with a certain name (e.g., 'measured_data.pkl'), the function will pick it up and open it directly. This process takes milliseconds.
    - If you want to change this, just change the name of the pickle in the pickle_path argument below.
- The pointers need to be pointed towards the files containing the data. Currently they point towards example data from Nepal to run this notebook.

### Datasets available
1. Nepal:
    - file1: "./input_data/nepal/measuredLandslides_all.csv", # All mapped landslides in region, with area, length_m, and width_m fields
    - file2: "./input_data/nepal/measuredLandslides_all_spatialStats.csv", # Zonal stats for all measured landslides
    - Spatial range:
        - Latitude: 27.6N - 28.4N
        - Longitude: 84.5E - 86.2E
2. New Zealand:
    - file1: "./input_data/nz/measuredLandslides_all.csv", # All mapped landslides in region, with area, length_m, and width_m fields
    - file2: "./input_data/nz/measuredLandslides_all_spatialStats.csv", # Zonal stats for all measured landslides
    - Spatial range:
        - Latitudes: (-41.7)N - (-43.1)N
        - Longitudes: 172.22E - 173.9E
3. Papua New Guinea:
    - file1: "./input_data/png/measuredLandslides_all.csv", # All mapped landslides in region, with area, length_m, and width_m fields
    - file2: "./input_data/png/measuredLandslides_all_spatialStats.csv", # Zonal stats for all measured landslides
    - Spatial range:
        - Latitudes: (-5.8)N - (-6.32)N
        - Longitudes: 142.2E - 143.4E
4. Japan:
    - file1: "./input_data/japan/measuredLandslides_all.csv", # All mapped landslides in region, with area, length_m, and width_m fields
    - file2: "./input_data/japan/measuredLandslides_all_spatialStats.csv", # Zonal stats for all measured landslides
    - Spatial range:
        - Latitudes: 38.82N - 39.19N
        - Longitudes: 140.7E - 140.95E

In [ ]:
# To choose which dataset to work with, select the region below.
# The default Nepal example uses the bundled DEM path above and does not need network access.
config['dem_info']['region'] = 'nepal'  # 'nepal', 'png', 'nz', 'japan'

# These lat-long bounds are used only when config['dem_info']['use_local_dem'] = False.
config['dem_info']['north'] = 28.29
config['dem_info']['east'] = 85.20
config['dem_info']['south'] = 28.18
config['dem_info']['west'] = 85.04


### Set up file paths, etc. for input and output

In [ ]:
# Dictionary of directory paths for each region
region_paths = {
    "nepal":  "./input_data/nepal",
    "png":  "./input_data/png/",
    "nz": "./input_data/nz/",
    "japan": "./input_data/japan/"
}
output_folders = {
    "nepal": "./output_data/nepal",
    "png": "./output_data/png",
    "nz": "./output_data/nz",
    "japan": "./output_data/japan"
}

region = config['dem_info']['region']
if region not in region_paths:
    raise ValueError(f"Unknown region '{region}'. Valid options: {list(region_paths.keys())}")

file_name_dict = {
    "file1": os.path.join(region_paths[region], "measuredLandslides_all.csv"),
    "file2": os.path.join(region_paths[region], "measuredLandslides_all_ZonalStats.csv")
}

pickle_path = os.path.join(region_paths[region], "measured_data.pkl")
config['output']['output_dir'] = output_folders[region]

print("Loaded paths for region:", region)

kde_dict = None
bundle = pickle_or_not_to_pickle(file_name_dict=file_name_dict,
                                    pickle_path=pickle_path, verbose=True)
kde_dict = {
    'kde_data': bundle.get('kde_data'),
    'kde_transform': bundle.get('kde_transform'),
}

logger = setup_logger(
        name="landslider",
        log_dir=config['output']['output_dir'],
        level=config.get("log_level", "INFO"),
        to_console=False,
    )
logger.info("=== Shallow Landslider Run Started ===")
logger.info(f"Using config: {config}")

print('Loaded measured KDEs.')

## Build Landlab grid, route flow, and derive terrain attributes
- Uses get_topo() and bmi-topography to download and set up a DEM from OpenTopo for you to help quickly set up a grid. If you have your own grid, set it up here.
- apply_soil_depth() helps set up a simulated soil depth distribution
    - Using a curvature-based distribution here requires richdem and calculate_terrain_attribute()
- If these already exist, you can skip the next several cells

In [ ]:
# Get DEM and build grid
dem_info = config['dem_info']

if dem_info.get('use_local_dem', True):
    local_dem_path = Path(dem_info['local_dem_path'])
    if not local_dem_path.exists():
        raise FileNotFoundError(f"Bundled DEM not found: {local_dem_path}")
    mg, z, _ = get_topo(
        buffer=dem_info['buffer'],
        dem_type=dem_info['dem_type'],
        smooth_num=dem_info['smooth_num'],
        load_dem=str(local_dem_path),
    )
else:
    if not dem_info.get('api_key'):
        raise RuntimeError(
            "Set OPENTOPOGRAPHY_API_KEY or switch config['dem_info']['use_local_dem'] to True."
        )
    mg, z, _ = get_topo(
        dem_type=dem_info['dem_type'],
        north=dem_info['north'],
        south=dem_info['south'],
        east=dem_info['east'],
        west=dem_info['west'],
        buffer=dem_info['buffer'],
        smooth_num=dem_info['smooth_num'],
        api_key=dem_info['api_key'],
    )

logger.info("Loaded DEM")
print(f"Grid loaded: {mg.shape[0]} rows x {mg.shape[1]} cols, {mg.number_of_nodes:,} nodes")


### Flow routing
- Fills in any voids in the DEM.
- Calculates upstream drainage area (Can be avoided, only required for drainage area-based soil depth)
- How much time this takes strongly depends on the size of the grid

In [ ]:
pf = PriorityFloodFlowRouter(
    grid=mg,
    flow_metric=config['flow_params']['flow_metric'],
    separate_hill_flow=config['flow_params']['separate_hill_flow'],
    depression_handler=config['flow_params']['depression_handling'],
    update_hill_depressions=config['flow_params']['update_hill_depressions'],
    accumulate_flow=config['flow_params']['accumulate_flow'],
)
pf.run_one_step()
logger.info("Flow routing complete")

### Calculate planform curvature

In [ ]:
# Terrain attribute example: planform curvature (for curvature‑based soil depth using richdem)
curv = calculate_terrain_attribute(
    grid=mg, field_name='topographic__elevation', attrib='planform_curvature'
)
logger.info("Planform curvature calculated")

### Add soil depth
- Also adds bedrock__elevation field to grid (also optional)

In [ ]:
if 'soil__depth' not in mg.at_node:
    soil_depth = mg.add_zeros('soil__depth', at='node')

soil_depth = apply_soil_depth(
    grid=mg,
    max_soil_depth=config['soil_params']['max_soil_depth'],
    distribution=config['soil_params']['distribution'],
    relationship=config['soil_params']['relationship'],
    decay_rate=config['soil_params']['decay_rate'],
    exponent=config['soil_params']['exponent'],
    drainage_transform=config['soil_params']['drainage_transform'],
    drainage_threshold=config['soil_params']['drainage_threshold'],
    drainage_power=config['soil_params']['drainage_power'],
    P0=config['soil_params']['P0'],
    h_star=config['soil_params']['h_star'],
    D=config['soil_params']['D'],
    h_min=config['soil_params']['h_min'],
    h_no_ss=config['soil_params']['h_no_ss'],
    plot=config['soil_params']['plot_soil'],
)

# Bedrock elevation from DEM minus soil thickness
if 'bedrock__elevation' not in mg.at_node:
    mg.add_zeros('bedrock__elevation', at='node', clobber=True)
mg.at_node['bedrock__elevation'][:] = mg.at_node['topographic__elevation'] - mg.at_node['soil__depth']

print(f'Applied {config['soil_params']['distribution']}-based soil depth distribution')

logger.info(f"Applied {config['soil_params']['distribution']} soil depth distribution")

### Calculate slopes and topographic aspect

In [ ]:
slopes_rad = mg.calc_slope_at_node(elevs='topographic__elevation')
slopes_deg = np.degrees(slopes_rad)
aspect_nodes = np.array(mg.calc_aspect_at_node(elevs='topographic__elevation',
                                                unit='degrees', ignore_closed_nodes=True))
aspect_nodes[mg.boundary_nodes] = np.nan
logger.info("Calculated slopes and aspect")

## Generate earthquake PGA grids
- Quickly sets up arrays for PGA_h and PGA_v using parameters set up above
- These can be "uniform", "circular", "square", "diamond", or "exponential" to test various patterns
- Can also be skipped if you have existing arrays of the same size as the grid

In [ ]:
pga_h, pga_v = generate_acceleration_grid(
    grid=mg,
    horizontal_max=config['pga']['horizontal_max'],
    vertical_max=config['pga']['vertical_max'],
    distribution=config['pga']['distribution'],
    plot_grids=config['pga']['plot_grids'],
)
logger.info("Created earthquake PGA arrays")

## Initialise and run the ShallowLandslider component
- Parameters taken from dictionary at the top of the notebook
- If you want to compare model outputs with different parameter sets, toggle 'save_output_pickle' to True, and the outputs for each run will be saved to disk.
    - The runs can then be compared later.

In [ ]:
# Initialise component
ls = ShallowLandslider(
    grid=mg,
    cohesion_eff=config['soil_params']['cohesion_eff'],
    angle_int_frict=config['soil_params']['angle_int_frict'],
    submerged_soil_proportion=config['soil_params']['submerged_soil_proportion'],
    pga_h=pga_h,
    pga_v=pga_v,
    pga_h_max=config['pga']['horizontal_max'],
    pga_v_max=config['pga']['vertical_max'],
    selection_method=config['simulation']['selection_method'],
    proportion_method=config['simulation']['proportion_method'],
    custom_proportion=config['simulation']['custom_proportion'],
    random_seed=config['simulation']['random_seed'],
    aspect_interval=config['simulation']['aspect_interval'],
    compute_displacement=config['simulation']['compute_displacement'],
    time_shaking=config['simulation']['time_shaking'],
    displacement_threshold=config['simulation']['displacement_threshold'],
    enable_runout=config['simulation']["enable_runout"],
    update_soil=config['simulation']["update_soil"],
    g=9.81,
    split_by_width_config=(None if kde_dict is None else {
        'kde_data': kde_dict.get('kde_data'),
        'kde_transform': kde_dict.get('kde_transform'),
        'convergence_threshold': 0.75,
        'min_region_size': 10,
        'max_iterations': 10,
        'width_threshold': 1.5,
    }),
    verbose=config['output']['verbose'],
)

In [ ]:
fig, axes = plt.subplots(1, 2, layout='constrained', figsize=(12, 5))
plt.sca(axes[0])
imshowhs_grid(
    ls.grid,
    'topographic__elevation',
    plot_type='DEM',
    allow_colorbar=True,
    cbar_or='vertical',
    ticks_km=True,
    cbar_loc='lower right',
    cbar_height=1.0,
    cbar_width=0.3,
)
axes[0].set_title('Input DEM')

plt.sca(axes[1])
imshowhs_grid(
    ls.grid,
    'topographic__elevation',
    plot_type='Drape1',
    drape1=np.ma.masked_invalid(ls.grid.at_node['soil__depth']),
    cmap='viridis',
    allow_colorbar=True,
    cbar_or='vertical',
    ticks_km=True,
    cbar_loc='lower right',
    cbar_height=1.0,
    cbar_width=0.3,
)
axes[1].set_title('Soil depth draped over hillshade')

plt.show()


In [ ]:
# Run component
soil_depth_before_landsliding = mg.at_node['soil__depth'].copy()
ls.run_one_step()

pickle_dir = os.path.join(config['output']['output_dir'], 'pickled_runs')
if config['output']['save_output_pickle']:
    save_model_run(ls=ls, config=config, output_dir=pickle_dir)

props = ls.results.get('group_properties')
if props is None or len(props) == 0:
    raise RuntimeError('No group properties table was computed.')

props_filtered = props.loc[props['selected']].copy()
measured_spatial_stats_900greater = bundle.get('measured_spatial_stats_900greater')
labels = mg.at_node['landslide__selected_labels'].copy().reshape(mg.shape)
labels_masked = np.ma.masked_where(labels == 0, labels)

summary = {
    'candidate_groups': int(len(props)),
    'selected_groups': int(len(props_filtered)),
    'selected_nodes': int(np.count_nonzero(mg.at_node['landslide__selected_labels'] > 0)),
    'unstable_nodes': int(np.count_nonzero(mg.at_node['landslide__unstable_mask'])),
}
if 'landslide__newmark_displacement' in mg.at_node:
    disp = mg.at_node['landslide__newmark_displacement']
    valid_disp = disp[np.isfinite(disp) & (disp > 0)]
    summary['positive_displacement_nodes'] = int(valid_disp.size)
    summary['median_positive_displacement_m'] = float(np.median(valid_disp)) if valid_disp.size else 0.0

print('Run summary')
for key, value in summary.items():
    print(f'  {key}: {value}')

props_filtered.head()


## Optional runout subcomponent

Runout redistributes failed soil downslope after Newmark displacement is computed. To run it through `ShallowLandslider`, first route flow with `PriorityFloodFlowRouter(..., separate_hill_flow=True)`, then initialize the component with `compute_displacement=True`, `enable_runout=True`, and `update_soil=True`. The runout step updates `soil__depth` in place and stores diagnostic erosion/deposition arrays on `ls._runout`.

The main example above keeps runout off by default. Set the runout flags in the config cell before initializing `ls`, or opt into the demonstration cell below.


In [ ]:
# Optional runout demonstration
# Set this to True if you want to execute a separate runout-enabled model run.
RUN_RUNOUT_EXAMPLE = False
runout_diagnostics = None

required_runout_fields = (
    'hill_flow__receiver_node',
    'hill_flow__receiver_proportions',
)
missing_runout_fields = [name for name in required_runout_fields if name not in mg.at_node]

if config['simulation']['enable_runout'] and config['simulation']['update_soil']:
    soil_change = mg.at_node['soil__depth'] - soil_depth_before_landsliding
    runout_diagnostics = {
        'erosion': ls._runout._last_erosion.copy(),
        'deposition': ls._runout._last_deposition.copy(),
        'soil_change': soil_change.copy(),
    }
    print('Runout was enabled in the main component run.')
elif RUN_RUNOUT_EXAMPLE:
    if missing_runout_fields:
        raise RuntimeError(
            'Run PriorityFloodFlowRouter with separate_hill_flow=True before runout. '
            f'Missing fields: {missing_runout_fields}'
        )

    soil_depth_before_runout = mg.at_node['soil__depth'].copy()
    ls_runout = ShallowLandslider(
        grid=mg,
        cohesion_eff=config['soil_params']['cohesion_eff'],
        angle_int_frict=config['soil_params']['angle_int_frict'],
        submerged_soil_proportion=config['soil_params']['submerged_soil_proportion'],
        pga_h=pga_h,
        pga_v=pga_v,
        pga_h_max=config['pga']['horizontal_max'],
        pga_v_max=config['pga']['vertical_max'],
        selection_method=config['simulation']['selection_method'],
        proportion_method=config['simulation']['proportion_method'],
        custom_proportion=config['simulation']['custom_proportion'],
        random_seed=config['simulation']['random_seed'],
        aspect_interval=config['simulation']['aspect_interval'],
        compute_displacement=True,
        time_shaking=config['simulation']['time_shaking'],
        displacement_threshold=config['simulation']['displacement_threshold'],
        enable_runout=True,
        update_soil=True,
        split_by_width_config=(None if kde_dict is None else {
            'kde_data': kde_dict.get('kde_data'),
            'kde_transform': kde_dict.get('kde_transform'),
            'convergence_threshold': 0.75,
            'min_region_size': 10,
            'max_iterations': 10,
            'width_threshold': 1.5,
        }),
        verbose=config['output']['verbose'],
    )
    ls_runout.run_one_step()
    soil_change = mg.at_node['soil__depth'] - soil_depth_before_runout
    runout_diagnostics = {
        'erosion': ls_runout._runout._last_erosion.copy(),
        'deposition': ls_runout._runout._last_deposition.copy(),
        'soil_change': soil_change.copy(),
    }
else:
    print('Runout example skipped. Set RUN_RUNOUT_EXAMPLE = True to execute it.')
    print('Required fields present:', not missing_runout_fields)
    print('Required component flags: compute_displacement=True, enable_runout=True, update_soil=True')

if runout_diagnostics is not None:
    print(f"Nodes with changed soil depth: {np.count_nonzero(np.abs(runout_diagnostics['soil_change']) > 0):,}")
    print(f"Total erosion: {np.sum(runout_diagnostics['erosion']):.6g} m-node")
    print(f"Total deposition: {np.sum(runout_diagnostics['deposition']):.6g} m-node")


In [ ]:
if runout_diagnostics is None:
    print('Runout diagnostics are not available. Enable runout or set RUN_RUNOUT_EXAMPLE = True above.')
else:
    fig, axes = plt.subplots(1, 2, layout='constrained', figsize=(12, 5))

    plt.sca(axes[0])
    imshowhs_grid(
        mg,
        'topographic__elevation',
        plot_type='Drape1',
        drape1=np.ma.masked_equal(runout_diagnostics['erosion'], 0),
        cmap='magma',
        allow_colorbar=True,
        cbar_or='vertical',
        ticks_km=True,
        cbar_loc='lower right',
        cbar_height=0.8,
        cbar_width=0.3,
    )
    axes[0].set_title('Runout erosion')

    plt.sca(axes[1])
    imshowhs_grid(
        mg,
        'topographic__elevation',
        plot_type='Drape1',
        drape1=np.ma.masked_equal(runout_diagnostics['deposition'], 0),
        cmap='viridis',
        allow_colorbar=True,
        cbar_or='vertical',
        ticks_km=True,
        cbar_loc='lower right',
        cbar_height=0.8,
        cbar_width=0.3,
    )
    axes[1].set_title('Runout deposition')

    plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, layout='constrained', figsize=(15, 5))

plt.sca(axes[0])
imshowhs_grid(
    mg,
    'topographic__elevation',
    plot_type='Drape1',
    drape1=labels_masked,
    cmap='tab20',
    allow_colorbar=False,
    ticks_km=True,
)
axes[0].set_title('Selected landslide labels')

plt.sca(axes[1])
imshowhs_grid(
    mg,
    'topographic__elevation',
    plot_type='Drape1',
    drape1=np.ma.masked_invalid(mg.at_node['landslide__factor_of_safety']),
    cmap='magma_r',
    allow_colorbar=True,
    cbar_or='vertical',
    ticks_km=True,
    cbar_loc='lower right',
    cbar_height=0.8,
    cbar_width=0.3,
)
axes[1].set_title('Factor of safety')

plt.sca(axes[2])
if 'landslide__newmark_displacement' in mg.at_node:
    displacement = np.ma.masked_less_equal(mg.at_node['landslide__newmark_displacement'], 0)
    imshowhs_grid(
        mg,
        'topographic__elevation',
        plot_type='Drape1',
        drape1=displacement,
        cmap='plasma',
        allow_colorbar=True,
        cbar_or='vertical',
        ticks_km=True,
        cbar_loc='lower right',
        cbar_height=0.8,
        cbar_width=0.3,
    )
    axes[2].set_title('Positive Newmark displacement')
else:
    axes[2].text(0.5, 0.5, 'Displacement disabled', ha='center', va='center')
    axes[2].set_axis_off()

plt.show()


## Publication-ready output figures

The following cells provide a compact set of outputs suitable for a public quickstart: selected landslide maps, stability and displacement fields, and observed-vs-modelled distributions for area, elevation, and slope.


## Visualise simulated landslides
- Plots a figure with four panels:
    1. Simulated landslides draped over hillshade
    2. Histogram and ecdf of landslide area (mapped vs. simulated)
    3. Histogram and ecdf of mean landslide elevations (mapped vs. simulated)
    4. Histogram and ecdf of mean landslide slopes (mapped vs. simulated)
- *Only* compares modelled data with all measured data, rather than data clipped to subregions.

In [ ]:
print(f"Selected groups: {len(props_filtered):,}")
if len(props_filtered) > 0:
    display_cols = [
        'area',
        'median_elevation',
        'median_slope',
        'mean_aspect',
        'slope_direction_length_new',
        'perpendicular_width_new',
    ]
    display(props_filtered[display_cols].describe().T)
else:
    print('No landslide groups selected for this parameter set.')


In [ ]:
plot_comparison_panels_with_ecdf(
    observed_df=measured_spatial_stats_900greater,
    model_df=props_filtered,
    mg=mg,                     # optional
    labels_masked=labels_masked,  # optional
    title=f"Modeled output: {config['soil_params']['distribution']} - {config['soil_params']['relationship']}",
    save_path=None             # or "comparison_panels.pdf"
)

## Compare different runs
- If 'save_output_pickle' was toggled to `True`, then you can run the next few cells to compare the results of different runs.

In [ ]:
# Optional saved-run comparison.
# This cell is useful only after running the notebook multiple times with
# config['output']['save_output_pickle'] = True.
if not config['output']['save_output_pickle'] or not Path(pickle_dir).exists():
    runs = {}
    model_dfs_by_run = {}
    print('No saved runs found. Set save_output_pickle=True and rerun to compare scenarios.')
else:
    runs = load_all_runs(pickle_dir)
    model_dfs_by_run = {
        str(name): run.get('group_properties')
        for name, run in runs.items()
        if isinstance(run, dict) and run.get('group_properties') is not None
    }
    model_dfs_by_run = {
        name: df.loc[df['selected']].copy() if 'selected' in df else df.copy()
        for name, df in model_dfs_by_run.items()
    }
    print(f'Loaded {len(model_dfs_by_run)} saved run(s).')

column_mapping = {
    'Area_m2': 'area',
    'Elevation_m_mean': 'median_elevation',
    'Slope_deg_mean': 'median_slope',
    'Aspect_deg_median': 'mean_aspect',
}


In [ ]:
if not model_dfs_by_run:
    print('Saved-run comparison skipped: no saved runs are available.')
else:
    metrics = [
        ('Area_m2', 'area', 'Area (m?)'),
        ('Elevation_m_mean', 'median_elevation', 'Elevation (m)'),
        ('Slope_deg_mean', 'median_slope', 'Slope (degrees)'),
    ]
    fig, axes = plt.subplots(1, len(metrics), figsize=(15, 4), layout='constrained')
    for ax, (obs_col, model_col, label) in zip(axes, metrics):
        if obs_col in measured_spatial_stats_900greater:
            ax.hist(measured_spatial_stats_900greater[obs_col].dropna(), bins=30, density=True, alpha=0.35, label='observed')
        for name, df in model_dfs_by_run.items():
            if model_col in df:
                ax.hist(df[model_col].dropna(), bins=30, density=True, histtype='step', linewidth=1.5, label=name)
        ax.set_xlabel(label)
        ax.set_ylabel('Density')
    axes[-1].legend(fontsize=8)
    plt.show()


In [ ]:
if not model_dfs_by_run:
    print('Performance summary skipped: no saved runs are available.')
else:
    rows = []
    for name, df in model_dfs_by_run.items():
        row = {'run': name, 'selected_groups': len(df)}
        for obs_col, model_col in column_mapping.items():
            if obs_col in measured_spatial_stats_900greater and model_col in df and len(df) > 0:
                row[f'{model_col}_median_ratio'] = (
                    np.nanmedian(df[model_col]) / np.nanmedian(measured_spatial_stats_900greater[obs_col])
                )
        rows.append(row)
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except ImportError:
        print(rows)


## Troubleshooting & Tips

- **Field names** follow the Landlab convention used in the component (e.g., `topographic__elevation`, `soil__depth`).
- If you see a `ValueError` about sizes, ensure arrays match `mg.number_of_nodes`.
- The public notebook uses a bundled DEM by default. To download a fresh DEM, set `use_local_dem=False` and provide `OPENTOPOGRAPHY_API_KEY` in your environment.
- Use `selection_method='pga_weighted'` for a deterministic proportion based on PGA; otherwise keep `'probabilistic'`.
- Turn on `compute_displacement=True` to write `landslide__newmark_displacement`.
- For runout, keep `flow_params.separate_hill_flow=True` and set `compute_displacement=True`, `enable_runout=True`, and `update_soil=True`.
- Set `aspect_interval` to a coarser (e.g., 45°) if too many subgroups are created.
- For reproducibility, set `random_seed`.
- For very small regions, consider post‑processing with `handle_small_regions` in your helper functions or merge/keep logic.

> For performance on large domains, start with smaller extents and lower `smooth_num`, and avoid plotting huge arrays.
